# 02 — Stability audit (provisional)

**Status: provisional.** The audit uses the inner 2016–17 period plus burned retrospective views (2018–22 and COVID 2020–21). It was **not re-run on the Sprint-A holdout**. Every number here is descriptive/provisional and must not be written as a confirmatory headline.

**What is audited.**

- Exact additive decomposition of the 49-feature logistic decision function.
- Source- and target-actor views, including their redundancy.
- Five dyad-cluster bootstrap refits ("seed" replicates with C fixed), not optimizer seeds.
- Period/regime and D2 window contrasts.
- Top-k churn and a decision proxy.
- HGB block occlusion as a model-class check.

**Not done.** No fitting, no new data, no ICEWS reads, no row-level predictions.


In [ ]:
from pathlib import Path
import json

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "results" / "framework_pilot").is_dir():
            return cand
    return here


ROOT = find_repo_root()
STABILITY = ROOT / "results" / "framework_pilot" / "stability"
TABLES = STABILITY / "tables"


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def load_json(path: Path) -> dict:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return {}
    with path.open(encoding="utf-8") as fh:
        return json.load(fh)


def load_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return pd.DataFrame()
    return pd.read_csv(path)


print("repo root:", ROOT)
print("stability artifacts:", "present" if STABILITY.is_dir() else "MISSING")


## Audit scope and provenance

The periods are displayed exactly as recorded in `summary.json`. "Burned retrospective" means the period had already been observed by earlier exploratory runs; it is not a fresh test.


In [ ]:
summary = load_json(STABILITY / "summary.json")

if summary:
    print("status:", summary.get("status"))
    print("sprint:", summary.get("sprint"))
    periods = pd.DataFrame(summary.get("scope", {}).get("periods", []))
    if not periods.empty:
        display(periods)
    exact = summary.get("exactness", {})
    print("exactness verdict:", exact.get("verdict"))
    print(
        "decision-function max abs error:",
        exact.get("decision_function_max_abs_error"),
    )
    print("source/target note:", summary.get("source_target", {}).get("interpretation"))
else:
    print("No stability summary available.")


## Exactness

The identity is **intercept + 49 feature contributions = decision function**. The maximum absolute error is 1.42×10⁻⁶ in log-odds. The larger raw-vs-logit error (~4×10⁻⁵) is a float32 `predict_proba` round-trip, not a decomposition failure. Temperature is T=1.0, so the temperature track coincides with the raw track; isotonic is excluded because it is not affine.


In [ ]:
exactness = load_csv(TABLES / "exactness_check.csv")

if not exactness.empty:
    etab = exactness[
        [
            "period", "n_rows", "decision_max_abs_error",
            "temperature_max_abs_error", "accumulator_max_abs_error",
            "accumulator_rows_above_tol",
        ]
    ].rename(
        columns={
            "period": "Period",
            "n_rows": "Rows",
            "decision_max_abs_error": "Decision max abs error",
            "temperature_max_abs_error": "Temperature max abs error",
            "accumulator_max_abs_error": "Stream max abs error",
            "accumulator_rows_above_tol": "Stream rows > 1e-6",
        }
    )
    display(etab)
else:
    print("Exactness table not available.")


## Actor buckets and source/target redundancy

The bucket shares are shares of the mean absolute contribution, not variance shares and not risk. Source and target actor views carry about 46% of the absolute contribution together. At the actor level they correlate at r ≈ 0.998, so they are **not independent evidence**; report both, but do not double-count them.

Positive attribution means "above the training mean", not "high absolute risk". The top positive actors are low-activity microstates with mean predicted probabilities far below the period base rate.


In [ ]:
buckets = load_csv(TABLES / "bucket_shares.csv")

if not buckets.empty:
    btab = buckets[buckets["view"] == "all_features"].pivot(
        index="bucket", columns="period", values="abs_share"
    )
    period_order = ["inner_2016_17", "retro_2018_22", "covid_2020_21"]
    btab = btab.reindex(
        columns=[p for p in period_order if p in btab.columns]
    )
    btab = btab.round(4)
    display(btab)
else:
    print("Bucket-share table not available.")

corr = pd.DataFrame(
    summary.get("source_target", {}).get("source_target_correlation", [])
)
if not corr.empty:
    ctab = corr[["period", "n", "pearson", "spearman", "mean_abs_diff"]].copy()
    for col in ["pearson", "spearman", "mean_abs_diff"]:
        ctab[col] = ctab[col].astype(float).round(4)
    display(ctab)
else:
    print("Source/target correlation table not available.")


## Variance decomposition: regime, not seed

These are within-actor variance shares. Seed noise is 4–5%; period/regime is 35–37%; the actor×time residual is 59–60%. The median regime/seed variance ratio is 9.4 (source) and 12.8 (target); 79–88% of actors have a ratio above 2. This is the central stability finding: the audit is **regime-limited**, not seed-limited.


In [ ]:
variance = load_csv(TABLES / "variance_decomposition.csv")

if not variance.empty:
    vtab = variance[variance["level"] == "within_actor_total"].pivot(
        index="component", columns="role", values="share"
    )
    vtab = vtab.round(4)
    display(vtab)

    ratio = variance[
        (variance["level"] == "actor_ratio")
        & (variance["component"].isin(
            ["regime_var_over_seed_var_median", "share_actors_ratio_gt_2"]
        ))
    ]
    if not ratio.empty:
        rtab = ratio.pivot(
            index="component", columns="role", values="share"
        )
        display(rtab.round(3))
else:
    print("Variance-decomposition table not available.")


## Top-k churn: cluster bootstrap, period, window

The five "seeds" are dyad-cluster bootstrap refits with C fixed, not optimizer seeds. Signed top-k Jaccard is the decision-relevant measure. At k=10, the mean seed-pair Jaccard is 0.91–1.00 (worst pair 0.80); period pairs fall to 0.54–0.82; D2 window pairs fall as low as 0.33. Rank correlation stays high (0.95–0.99), but the **exact membership** of a 10-actor review set is not stable across regimes or windows.

Cold-start/onset explanations should not be pooled with stable-history cases.


In [ ]:
churn = load_csv(TABLES / "topk_churn_actor_month.csv")

if not churn.empty:
    ctab = churn[churn["k"] == 10][
        [
            "period", "role", "score", "seed_pair_mean_jaccard",
            "seed_pair_min", "mean_churn",
            "median_boundary_gap_over_seed_sd",
        ]
    ].rename(
        columns={
            "period": "Period",
            "role": "View",
            "score": "Score",
            "seed_pair_mean_jaccard": "Seed Jaccard@10",
            "seed_pair_min": "Seed Jaccard@10 min",
            "mean_churn": "Mean churn",
            "median_boundary_gap_over_seed_sd": "Boundary gap / seed SD",
        }
    )
    for col in ["Seed Jaccard@10", "Seed Jaccard@10 min", "Mean churn"]:
        ctab[col] = ctab[col].astype(float).round(3)
    ctab = ctab.reset_index(drop=True)
    display(ctab)
else:
    print("Top-k churn table not available.")

periods = load_csv(TABLES / "stability_periods.csv")
if not periods.empty:
    ptab = periods[
        ["role", "period_a", "period_b", "spearman",
         "jaccard_pos@10", "sign_agreement"]
    ].rename(
        columns={
            "role": "View",
            "period_a": "Period A",
            "period_b": "Period B",
            "spearman": "Spearman",
            "jaccard_pos@10": "Signed Jaccard@10",
            "sign_agreement": "Sign agreement",
        }
    )
    for col in ["Spearman", "Signed Jaccard@10", "Sign agreement"]:
        ptab[col] = ptab[col].astype(float).round(3)
    ptab = ptab.reset_index(drop=True)
    display(ptab)
else:
    print("Period-stability table not available.")


In [ ]:
windows = load_csv(TABLES / "stability_windows.csv")

if not windows.empty:
    wtab = (
        windows.groupby(["period", "role", "window_a", "window_b"])
        .agg(
            spearman=("spearman", "median"),
            jaccard_pos10=("jaccard_pos@10", "median"),
            sign_agreement=("sign_agreement", "median"),
        )
        .reset_index()
        .rename(
            columns={
                "period": "Period",
                "role": "View",
                "window_a": "Window A",
                "window_b": "Window B",
                "spearman": "Spearman (median)",
                "jaccard_pos10": "Signed Jaccard@10 (median)",
                "sign_agreement": "Sign agreement (median)",
            }
        )
    )
    for col in [
        "Spearman (median)", "Signed Jaccard@10 (median)",
        "Sign agreement (median)",
    ]:
        wtab[col] = wtab[col].astype(float).round(3)
    wtab = wtab.reset_index(drop=True)
    display(wtab)
else:
    print("Window-stability table not available.")


## Decision proxy: risk selects, attribution explains

The risk alert ranks actor-months by mean predicted probability. The attribution alert ranks them by the actor's own bucket contribution. These are different objects. Across the reported k values, risk lift is 2.5–4.3 for actor-month review and 7.9–9.1 for dyad review, with near-unit seed Jaccard. Attribution lift is 0.10–0.20 (actor-month) and 0.06–0.56 (dyad), with essentially zero overlap with the risk alert.

The attribution top-k is therefore **anti-useful as an alert rule**. That is a negative decision-impact result, not a failed stability result. Use risk to choose what to review, then attach the exact decomposition to explain why.


In [ ]:
decision = load_csv(TABLES / "decision_impact_actor_month.csv")

if not decision.empty:
    dtab = decision[decision["k"] == 10][
        [
            "period", "role", "alert", "mean_lift_vs_month_base",
            "lift_ci_lo", "lift_ci_hi",
            "mean_jaccard_with_risk_alert",
        ]
    ].rename(
        columns={
            "period": "Period",
            "role": "View",
            "alert": "Alert",
            "mean_lift_vs_month_base": "Mean lift",
            "lift_ci_lo": "Lift CI low",
            "lift_ci_hi": "Lift CI high",
            "mean_jaccard_with_risk_alert": "Jaccard with risk",
        }
    )
    for col in ["Mean lift", "Lift CI low", "Lift CI high", "Jaccard with risk"]:
        dtab[col] = dtab[col].astype(float).round(3)
    dtab = dtab.reset_index(drop=True)
    display(dtab)
else:
    print("Actor-month decision table not available.")

dyad = load_csv(TABLES / "decision_impact_dyad.csv")
if not dyad.empty:
    ytab = dyad[dyad["k"].isin([10, 100, 1000])][
        [
            "alert", "k", "precision", "recall", "lift_vs_base",
            "jaccard_with_risk",
        ]
    ].rename(
        columns={
            "alert": "Alert",
            "k": "k",
            "precision": "Precision",
            "recall": "Recall",
            "lift_vs_base": "Lift",
            "jaccard_with_risk": "Jaccard with risk",
        }
    )
    for col in ["Precision", "Recall", "Lift", "Jaccard with risk"]:
        ytab[col] = ytab[col].astype(float).round(3)
    ytab = ytab.reset_index(drop=True)
    display(ytab)
else:
    print("Dyad decision table not available.")


## HGB block occlusion

HGB is a separate refit (150 iterations, current graph cache), not the 120-iteration holdout capacity control. Occlusion–occlusion Spearman is 0.69–0.82: broad rank agreement. At actor level, top-10 absolute overlap is only 0.05–0.25, so model-class disagreement is concentrated in the tail. This is descriptive agreement, not an HGB-vs-logistic superiority or parity claim.


In [ ]:
hgb = load_csv(TABLES / "hgb_occlusion_agreement.csv")

if not hgb.empty:
    htab = hgb[hgb["level"] == "actor"][
        [
            "period", "role", "spearman_logocc_hgb",
            "spearman_logocc_ci_lo", "spearman_logocc_ci_hi",
            "sign_logocc_hgb", "jaccard_abs@10",
            "spearman_abs_attr_hgb",
        ]
    ].rename(
        columns={
            "period": "Period",
            "role": "View",
            "spearman_logocc_hgb": "Occlusion-occlusion Spearman",
            "spearman_logocc_ci_lo": "CI low",
            "spearman_logocc_ci_hi": "CI high",
            "sign_logocc_hgb": "Sign agreement",
            "jaccard_abs@10": "Top-10 abs occlusion Jaccard",
            "spearman_abs_attr_hgb": "Abs attribution vs HGB",
        }
    )
    for col in [
        "Occlusion-occlusion Spearman", "CI low", "CI high",
        "Sign agreement", "Top-10 abs occlusion Jaccard",
        "Abs attribution vs HGB",
    ]:
        htab[col] = htab[col].astype(float).round(3)
    htab = htab.reset_index(drop=True)
    display(htab)
else:
    print("HGB occlusion table not available.")


## Provisional bottom line

- Exactness **passes**: the additive decomposition reproduces the decision function to 1.42×10⁻⁶.
- Actor views are **stable to cluster resampling** but **regime-limited**: seed noise is 4–5% of within-actor variance, regime is 35–37%.
- Source and target views are **not independent** (r ≈ 0.998).
- Attribution top-k is **not an alert rule**; risk is. Post-hoc explanation inside a risk alert is stable (Spearman ≥0.996).
- The audit was **not re-run on the holdout**. None of these numbers is a confirmatory holdout result.
